# 04 - Results: figures and tables

Reads the stored results of notebooks 02 and 03 and produces every remaining
figure and table in the paper. No model is refitted here.

* **Figure 4** - out-of-sample RMSE, models x information sets
* **Figure 5** - directional accuracy, models x information sets
* **Figure 6** - Extra Trees on the Combined set: predicted vs observed
* **Table 2** and Section 4.3 - Diebold-Mariano tests (`dm_tests.csv`)
* Sections 4.2-4.5 - every metric quoted in the text (`all_metrics.csv`, `best_model_per_set.csv`)
* `paper_tables.xlsx` - all tables in one workbook

In [ ]:
# Make the project package importable, from this folder or the project root.
# In Colab: mount Drive and set ROOT = Path("/content/drive/MyDrive/NEPSE_replication") instead.
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "nepse").is_dir())
sys.path.insert(0, str(ROOT / "src"))
print("project root:", ROOT)

In [ ]:
import pandas as pd
from nepse import config, experiments, metrics, plots

config.ensure_dirs()
plots.apply_style()

metrics_long, predictions, walkforward = experiments.load_results()
observed = predictions["observed"]
baseline = metrics.zero_return_rmse_bps(observed.values)

print(f"{len(metrics_long)} model x information-set results")
print(f"test period: {observed.index.min().date()} to {observed.index.max().date()} "
      f"({len(observed)} days)")
print(f"zero-return baseline RMSE: {baseline:.2f} bps")

## Figures 4 and 5

The heatmaps show five information sets (`config.PAPER_INFO_ORDER`); the PLS
variant is left out for readability but is in every table.

In [ ]:
rmse_matrix = experiments.metric_matrix(metrics_long, "RMSE_bps",
                                       info_order=config.PAPER_INFO_ORDER)
dir_matrix = experiments.metric_matrix(metrics_long, "DirAcc",
                                      info_order=config.PAPER_INFO_ORDER)

_ = plots.metric_heatmap(rmse_matrix, "Out-of-sample RMSE (bps)",
                         "fig4_rmse_heatmap", fmt="{:.1f}", lower_better=True)
_ = plots.metric_heatmap(dir_matrix, "Directional accuracy",
                         "fig5_diracc_heatmap", fmt="{:.3f}", lower_better=False)

## Figure 6 - best model overall, predicted vs observed

In [ ]:
full_rmse = experiments.metric_matrix(metrics_long, "RMSE_bps")
best_model, best_info = full_rmse.stack().idxmin()
print(f"best overall: {best_model} on {config.INFO_LABELS[best_info]} "
      f"({full_rmse.loc[best_model, best_info]:.2f} bps)")

model, info = config.DIAG_MODEL, config.DIAG_SET          # ET on Combined
row = metrics_long[(metrics_long["model"] == model) & (metrics_long["info"] == info)].iloc[0]
_ = plots.pred_vs_obs(observed.index, observed.values, predictions[f"{info}|{model}"].values,
                      model, config.INFO_LABELS[info], row,
                      name=f"fig6_pred_vs_obs_{model}_{info}")

## Table 2 - Diebold-Mariano tests

Each information set against NEPSE-only with the model held fixed, for Extra Trees
(Table 2) and gradient boosting (Section 4.3), under squared and absolute loss.
A positive statistic means the information set has the lower loss.

In [ ]:
dm = pd.concat([experiments.dm_table(predictions, model=m) for m in ("ET", "GBR")],
               ignore_index=True)
dm.round(4).to_csv(config.TABLE_DIR / "dm_tests.csv", index=False)
print(dm.round(3).to_string(index=False))

## Other numbers quoted in the text

In [ ]:
best = experiments.best_per_set(metrics_long, predictions, dm_model="ET")
all_metrics = metrics_long.sort_values(["info", "RMSE_bps"]).round(4)
best.to_csv(config.TABLE_DIR / "best_model_per_set.csv")
all_metrics.to_csv(config.TABLE_DIR / "all_metrics.csv", index=False)

print(f"Zero-return baseline: {baseline:.2f} bps\n")
print(best.to_string())
if walkforward is not None:
    print("\nWalk-forward RMSE (bps), gradient boosting, mean and s.d. across origins")
    print(walkforward.round(2).to_string())

In [ ]:
with pd.ExcelWriter(config.TABLE_DIR / "paper_tables.xlsx", engine="openpyxl") as xl:
    dm.round(4).to_excel(xl, sheet_name="Table2_DieboldMariano", index=False)
    rmse_matrix.rename(columns=config.INFO_LABELS).round(2).to_excel(xl, sheet_name="Fig4_RMSE")
    dir_matrix.rename(columns=config.INFO_LABELS).round(3).to_excel(xl, sheet_name="Fig5_DirAcc")
    best.to_excel(xl, sheet_name="Best_model_per_set")
    all_metrics.to_excel(xl, sheet_name="All_metrics", index=False)
    if walkforward is not None:
        walkforward.round(3).to_excel(xl, sheet_name="Walk_forward")
    for name in ("coverage_screen", "leadlag_ranking"):
        path = config.TABLE_DIR / f"{name}.csv"
        if path.exists():
            pd.read_csv(path).to_excel(xl, sheet_name=name, index=False)
    pd.DataFrame({"zero_return_RMSE_bps": [baseline]}).to_excel(
        xl, sheet_name="Baseline", index=False)
print("[saved] results/tables/paper_tables.xlsx")